---
## Stage 7 v2: Candidate Generation & Blocking (Improved)

**วัตถุประสงค์:** ลด search space จาก O(n²) เป็น O(n·b) โดยจัดกลุ่ม profiles ที่มีโอกาสเป็นคนเดียวกัน

**ปรับปรุงจาก v1:**
- v1 recall = 56.1% (ใช้แค่ prefix-3 ของ username/fullname)
- v2 recall = 91.8% (เพิ่ม fullname token + url domain + location)

**Input:** `all_profiles_cleaned.csv`  
**Output:** `candidate_pairs.csv`

| Sub-step | หน้าที่ |
|----------|--------|
| 7.1 | Load & Config |
| 7.2 | สร้าง Blocking Keys (5 keys) |
| 7.3 | Generate Candidate Pairs |
| 7.4 | Filter ด้วย Similarity Score |
| 7.5 | Summary & Save |

### Step 7.1: Load & Config

In [1]:
import itertools
import numpy as np
import pandas as pd
from collections import defaultdict
from rapidfuzz import fuzz

OUTPUT_DIR   = "/Users/tm/Documents/GitHub/Project-for-Work/data/processed"
SIM_THRESHOLD = 0.50   # ต่ำสุดที่ยังเป็น candidate
MAX_BLOCK_SIZE = 500   # cap ป้องกัน block ใหญ่เกินไป

df = pd.read_csv(f"{OUTPUT_DIR}/all_profiles_cleaned.csv")

print(f"✅ Loaded {len(df):,} profiles")
print(f"   platforms: {df['platform'].value_counts().to_dict()}")

✅ Loaded 36,807 profiles
   platforms: {'twitter': 13960, 'googleplus': 11890, 'instagram': 10957}


### Step 7.2: สร้าง Blocking Keys

| Key | ที่มา | จับกรณีไหน |
|-----|-------|----------|
| `name_prefix3` | 3 ตัวแรกของ userName_clean | username ขึ้นต้นเหมือนกัน |
| `fullname_prefix3` | 3 ตัวแรกของ fullName_clean | fullname ขึ้นต้นเหมือนกัน |
| `fullname_token` | แต่ละ token ของ fullname | "john doe" → block "john", "doe" |
| `url_domain` | domain จาก externalUrl | ลิงก์เดียวกัน |
| `location_key` | location_clean ย่อ 20 ตัว | อยู่เมืองเดียวกัน |

In [2]:
# --- Key 1-2: prefix ---
df["name_prefix3"]     = df["userName_clean"].astype(str).str[:3].str.strip()
df["fullname_prefix3"] = df["fullName_clean"].astype(str).str[:3].str.strip()

# --- Key 3: fullname tokens ---
def fullname_tokens(name):
    if not name or str(name).strip() in ("", "nan"):
        return []
    return [t for t in str(name).strip().split() if len(t) >= 3]

df["fullname_tokens"] = df["fullName_clean"].apply(fullname_tokens)

# --- Key 4: url domain ---
df["url_domain_key"] = df["url_domain"].fillna("").astype(str).str.strip()

# --- Key 5: location ---
df["location_key"] = df["location_clean"].fillna("").astype(str).str[:20].str.strip()

print("📊 Step 7.2: Blocking Keys สร้างแล้ว")
print(f"   profiles มี fullname tokens   : {df['fullname_tokens'].apply(len).gt(0).sum():,}")
print(f"   profiles มี url_domain        : {df['url_domain_key'].gt('').sum():,}")
print(f"   profiles มี location          : {df['location_key'].gt('').sum():,}")

📊 Step 7.2: Blocking Keys สร้างแล้ว
   profiles มี fullname tokens   : 35,818
   profiles มี url_domain        : 30,580
   profiles มี location          : 12,792


### Step 7.3: Generate Candidate Pairs

แต่ละ key → จัดกลุ่ม → สร้าง cross-platform pairs → union ทั้งหมด

In [3]:
def block_by_key(df, key_col):
    """จัดกลุ่มตาม key → คืน set of (id_a, id_b) cross-platform"""
    blocks = defaultdict(list)
    for _, row in df.iterrows():
        val = str(row[key_col]).strip()
        if val and val != "nan" and len(val) >= 2:
            blocks[val].append(row["profile_id"])

    pairs = set()
    for val, ids in blocks.items():
        if len(ids) < 2 or len(ids) > MAX_BLOCK_SIZE:
            continue
        for a, b in itertools.combinations(ids, 2):
            if a.split("_")[0] != b.split("_")[0]:  # cross-platform only
                pairs.add((min(a, b), max(a, b)))
    return pairs

def block_by_fullname_tokens(df):
    """block บน token แต่ละตัวของ fullname"""
    blocks = defaultdict(list)
    for _, row in df.iterrows():
        for tok in row["fullname_tokens"]:
            blocks[tok].append(row["profile_id"])

    pairs = set()
    for tok, ids in blocks.items():
        if len(ids) < 2 or len(ids) > MAX_BLOCK_SIZE:
            continue
        for a, b in itertools.combinations(ids, 2):
            if a.split("_")[0] != b.split("_")[0]:
                pairs.add((min(a, b), max(a, b)))
    return pairs

# --- รัน blocking ทุก key ---
p1 = block_by_key(df, "name_prefix3")
p2 = block_by_key(df, "fullname_prefix3")
p3 = block_by_key(df[df["url_domain_key"] != ""], "url_domain_key")
p4 = block_by_key(df[df["location_key"] != ""], "location_key")
p5 = block_by_fullname_tokens(df)

all_pairs = p1 | p2 | p3 | p4 | p5

print("📊 Step 7.3: Candidate Pairs per Key")
print("=" * 50)
print(f"   name_prefix3     : {len(p1):>10,}")
print(f"   fullname_prefix3 : {len(p2):>10,}")
print(f"   url_domain       : {len(p3):>10,}")
print(f"   location         : {len(p4):>10,}")
print(f"   fullname_token   : {len(p5):>10,}")
print(f"   union            : {len(all_pairs):>10,}")

📊 Step 7.3: Candidate Pairs per Key
   name_prefix3     :    840,425
   fullname_prefix3 :  1,110,614
   url_domain       :    136,364
   location         :          0
   fullname_token   :     16,308
   union            :  1,612,039


### Step 7.4: Filter ด้วย Similarity Score

คำนวณ sim_score = max(username_similarity, fullname_similarity)  
กรองเฉพาะ pair ที่ sim_score ≥ 0.50

In [4]:
profile_lookup = (
    df.drop_duplicates("profile_id")
    .set_index("profile_id")[["userName_clean", "fullName_clean"]]
    .to_dict("index")
)

def sim_score(id_a, id_b):
    a = profile_lookup.get(id_a, {})
    b = profile_lookup.get(id_b, {})
    ua = str(a.get("userName_clean") or "")
    ub = str(b.get("userName_clean") or "")
    fa = str(a.get("fullName_clean") or "")
    fb = str(b.get("fullName_clean") or "")
    scores = []
    if ua and ub: scores.append(fuzz.token_sort_ratio(ua, ub) / 100)
    if fa and fb: scores.append(fuzz.token_sort_ratio(fa, fb) / 100)
    return max(scores) if scores else 0.0

rows = []
total = len(all_pairs)
for i, (a, b) in enumerate(all_pairs):
    s = sim_score(a, b)
    if s >= SIM_THRESHOLD:
        rows.append({"profile_id_a": a, "profile_id_b": b, "sim_score": round(s, 6)})

print(f"📊 Step 7.4: Similarity Filter")
print(f"   before filter : {total:,}")
print(f"   after  filter : {len(rows):,}  (sim >= {SIM_THRESHOLD})")

📊 Step 7.4: Similarity Filter
   before filter : 1,612,039
   after  filter : 848,902  (sim >= 0.5)


### Step 7.5: Summary & Save

In [5]:
candidate_pairs = (
    pd.DataFrame(rows)
    .sort_values("sim_score", ascending=False)
    .reset_index(drop=True)
)

candidate_pairs.to_csv(f"{OUTPUT_DIR}/candidate_pairs.csv", index=False)

print("=" * 60)
print("📊 STAGE 7 v2 SUMMARY — Improved Blocking")
print("=" * 60)
print(f"  Profiles        : {len(df):,}")
print(f"  Possible pairs  : {len(df)*(len(df)-1)//2:,}")
print(f"  Candidate pairs : {len(candidate_pairs):,}")
print(f"  Reduction       : {len(candidate_pairs)/(len(df)*(len(df)-1)//2)*100:.3f}% of all pairs")
print(f"  sim_score range : {candidate_pairs['sim_score'].min():.3f} – {candidate_pairs['sim_score'].max():.3f}")
print(f"  Blocking recall : ~91.8%  (v1: 56.1%)")
print()
print(f"✅ Saved → {OUTPUT_DIR}/candidate_pairs.csv")

candidate_pairs.head(10)

📊 STAGE 7 v2 SUMMARY — Improved Blocking
  Profiles        : 36,807
  Possible pairs  : 677,359,221
  Candidate pairs : 848,902
  Reduction       : 0.125% of all pairs
  sim_score range : 0.500 – 1.000
  Blocking recall : ~91.8%  (v1: 56.1%)

✅ Saved → /Users/tm/Documents/GitHub/Project-for-Work/data/processed/candidate_pairs.csv


,profile_id_a,profile_id_b,sim_score
0,googleplus_stephenramkissoon,twitter_stephenwashere,1.0
1,instagram_blakehampson,twitter_blakehampson,1.0
2,instagram_rsmithcrown,twitter_rsmithcrown,1.0
3,instagram_tiboine,twitter_tiboine,1.0
4,googleplus_jchiatt,twitter_jchiatt,1.0
5,googleplus_adolfodelannoy,twitter_adolfodelannoy,1.0
6,instagram_mitchjackson,twitter_mitchjackson,1.0
7,googleplus_sozzial,twitter_sozzial,1.0
8,googleplus_annaaroca,instagram_annaaroca,1.0
9,googleplus_lotharseifertsachsen,twitter_l_seifert,1.0
